# Notebook 04: Text Embeddings

**Goal:** Generate semantic embeddings from anime descriptions for content-based similarity in our recommendation system.

**Approach:** TF-IDF + Genre/Theme Hybrid Embeddings
- Text embeddings (TF-IDF): Captures description semantics
- Metadata embeddings (Genre/Theme): Captures categorical content
- Hybrid combination: Best of both worlds

**Output:** 
- `embeddings_text.npy` - Final embeddings (1073 dimensions)
- Ready for FAISS indexing and similarity search

**Steps:**
1. Load and prepare data
2. Text preprocessing and cleaning
3. Generate TF-IDF embeddings
4. Combine with genre/theme features
5. Test embedding quality
6. Save embeddings

---

## 1. Setup and Load Data

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

DATA_DIR = Path('data')
PROCESSED_DIR = DATA_DIR / 'processed'

df = pd.read_parquet(PROCESSED_DIR / 'anime_features.parquet')

print("Data loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nText columns:")
print(f"  title: {df['title'].notna().sum():,} non-null")
print(f"  description: {df['description'].notna().sum():,} non-null")
print(f"  Avg description length: {df['description_length'].mean():.0f} characters")
print(f"\nMetadata features:")
print(f"  Genres: {len([c for c in df.columns if c.startswith('genre_')])}")
print(f"  Themes: {len([c for c in df.columns if c.startswith('theme_')])}")

Data loaded successfully
Shape: (19931, 189)

Text columns:
  title: 19,931 non-null
  description: 19,931 non-null
  Avg description length: 416 characters

Metadata features:
  Genres: 21
  Themes: 52


## 2. Text Preprocessing

Clean and prepare text data for optimal TF-IDF embedding generation.

In [7]:
def clean_text(text):
    """Clean and normalize text for embedding generation"""
    if pd.isna(text) or text == '':
        return ''
    
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s\-]', ' ', text)
    text = ' '.join(text.split())
    
    return text

print("Preprocessing text data...")

df['title_clean'] = df['title'].fillna('').apply(clean_text)
df['description_clean'] = df['description'].fillna('').apply(clean_text)

df['combined_text'] = (
    df['title_clean'] + ' ' + 
    df['title_clean'] + ' ' +
    df['description_clean'] + ' ' +
    df['Genres'].fillna('').apply(clean_text) + ' ' +
    df['Themes'].fillna('').apply(clean_text)
)

df['combined_text'] = df['combined_text'].str.strip()
df.loc[df['combined_text'] == '', 'combined_text'] = 'unknown anime'

print("Text preprocessing complete")
print(f"  Avg combined text length: {df['combined_text'].str.len().mean():.0f} chars")
print(f"  Min length: {df['combined_text'].str.len().min()}")
print(f"  Max length: {df['combined_text'].str.len().max()}")
print(f"\nSample cleaned text:")
print(f"  {df['combined_text'].iloc[0][:150]}...")

Preprocessing text data...
Text preprocessing complete
  Avg combined text length: 485 chars
  Min length: 15
  Max length: 3719

Sample cleaned text:
  cowboy bebop cowboy bebop crime is timeless by the year 2071 humanity has expanded across the galaxy filling the surface of other planets with settlem...


## 3. Generate TF-IDF Text Embeddings

Create text embeddings using TF-IDF with optimized parameters.

In [8]:
print("Generating TF-IDF embeddings...")
print("="*60)

tfidf = TfidfVectorizer(
    max_features=1000,
    min_df=3,
    max_df=0.7,
    ngram_range=(1, 3),
    stop_words='english',
    lowercase=True,
    strip_accents='unicode',
    sublinear_tf=True
)

print("TF-IDF Parameters:")
print(f"  max_features: 1000 (dimensionality)")
print(f"  min_df: 3 (ignore rare words)")
print(f"  max_df: 0.7 (ignore common words)")
print(f"  ngram_range: (1, 3) (uni/bi/trigrams)")
print(f"  sublinear_tf: True (dampens frequency)")

embeddings_tfidf = tfidf.fit_transform(df['combined_text']).toarray()
embeddings_text = normalize(embeddings_tfidf, axis=1)

print(f"\nText embeddings generated:")
print(f"  Shape: {embeddings_text.shape}")
print(f"  Sparsity: {(embeddings_text == 0).sum() / embeddings_text.size * 100:.1f}%")
print(f"  Memory: {embeddings_text.nbytes / (1024**2):.2f} MB")

feature_names = tfidf.get_feature_names_out()
print(f"\nVocabulary size: {len(feature_names)}")
print(f"Top 15 features: {feature_names[:15].tolist()}")

Generating TF-IDF embeddings...
TF-IDF Parameters:
  max_features: 1000 (dimensionality)
  min_df: 3 (ignore rare words)
  max_df: 0.7 (ignore common words)
  ngram_range: (1, 3) (uni/bi/trigrams)
  sublinear_tf: True (dampens frequency)

Text embeddings generated:
  Shape: (19931, 1000)
  Sparsity: 97.6%
  Memory: 152.06 MB

Vocabulary size: 1000
Top 15 features: ['10', '12', '13', '2nd', '2nd season', '3rd', '3rd season', 'abandoned', 'abilities', 'ability', 'able', 'academy', 'accident', 'accidentally', 'action']


## 3. Generate OPTIMAL Text Embeddings

**Maximizing Quality:** We'll create the best possible embeddings by:
1. Higher dimensionality (2000 features)
2. Character n-grams (captures partial words, typos)
3. Multiple TF-IDF variants (different perspectives)
4. Weighted ensemble of embeddings

In [9]:
print("GENERATING OPTIMAL TEXT EMBEDDINGS")
print("="*70)

# Strategy 1: Word-based TF-IDF (semantic meaning)
print("\n1. Word-based TF-IDF (2000 features, trigrams)...")
tfidf_word = TfidfVectorizer(
    max_features=2000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 3),
    stop_words='english',
    sublinear_tf=True,
    norm='l2'
)
emb_word = tfidf_word.fit_transform(df['combined_text']).toarray()
emb_word = normalize(emb_word, axis=1)
print(f"   Shape: {emb_word.shape}, Sparsity: {(emb_word==0).sum()/emb_word.size*100:.1f}%")

# Strategy 2: Character n-grams (captures typos, partial matches)
print("\n2. Character n-gram TF-IDF (1000 features, 3-5 chars)...")
tfidf_char = TfidfVectorizer(
    max_features=1000,
    analyzer='char',
    ngram_range=(3, 5),
    min_df=3,
    sublinear_tf=True,
    norm='l2'
)
emb_char = tfidf_char.fit_transform(df['combined_text']).toarray()
emb_char = normalize(emb_char, axis=1)
print(f"   Shape: {emb_char.shape}, Sparsity: {(emb_char==0).sum()/emb_char.size*100:.1f}%")

# Strategy 3: Title-focused embeddings (title is highly informative)
print("\n3. Title-focused embeddings (500 features)...")
tfidf_title = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    norm='l2'
)
emb_title = tfidf_title.fit_transform(df['title_clean']).toarray()
emb_title = normalize(emb_title, axis=1)
print(f"   Shape: {emb_title.shape}, Sparsity: {(emb_title==0).sum()/emb_title.size*100:.1f}%")

# Weighted ensemble
print("\n4. Creating weighted ensemble...")
print("   Weights: 50% word, 25% char, 25% title")

embeddings_text = np.concatenate([
    emb_word * 0.50,
    emb_char * 0.25,
    emb_title * 0.25
], axis=1)

embeddings_text = normalize(embeddings_text, axis=1)

print(f"\nFinal text embeddings:")
print(f"  Shape: {embeddings_text.shape}")
print(f"  Total dimensions: {embeddings_text.shape[1]}")
print(f"  Sparsity: {(embeddings_text == 0).sum() / embeddings_text.size * 100:.1f}%")
print(f"  Memory: {embeddings_text.nbytes / (1024**2):.2f} MB")

GENERATING OPTIMAL TEXT EMBEDDINGS

1. Word-based TF-IDF (2000 features, trigrams)...
   Shape: (19931, 2000), Sparsity: 98.5%

2. Character n-gram TF-IDF (1000 features, 3-5 chars)...
   Shape: (19931, 1000), Sparsity: 72.6%

3. Title-focused embeddings (500 features)...
   Shape: (19931, 500), Sparsity: 99.6%

4. Creating weighted ensemble...
   Weights: 50% word, 25% char, 25% title

Final text embeddings:
  Shape: (19931, 3500)
  Total dimensions: 3500
  Sparsity: 91.3%
  Memory: 532.22 MB


## 4. Create Hybrid Embeddings (Text + Metadata)

Combine text embeddings with genre/theme features for maximum recommendation quality.

In [10]:
print("CREATING HYBRID EMBEDDINGS")
print("="*70)

# Extract genre and theme features
genre_cols = [c for c in df.columns if c.startswith('genre_')]
theme_cols = [c for c in df.columns if c.startswith('theme_')]

genre_theme_features = df[genre_cols + theme_cols].values.astype(float)
genre_theme_normalized = normalize(genre_theme_features, axis=1)

print(f"\nMetadata features:")
print(f"  Genres: {len(genre_cols)}")
print(f"  Themes: {len(theme_cols)}")
print(f"  Total: {genre_theme_normalized.shape[1]} dimensions")
print(f"  Sparsity: {(genre_theme_normalized == 0).sum() / genre_theme_normalized.size * 100:.1f}%")

# Create hybrid embeddings with optimal weighting
# Text: 75% (most informative)
# Metadata: 25% (ensures genre/theme alignment)
print("\nCombining text + metadata...")
print("  Text weight: 75%")
print("  Metadata weight: 25%")

embeddings_hybrid = np.concatenate([
    embeddings_text * 0.75,
    genre_theme_normalized * 0.25
], axis=1)

embeddings_final = normalize(embeddings_hybrid, axis=1)

print(f"\nFinal hybrid embeddings:")
print(f"  Text dimensions: {embeddings_text.shape[1]}")
print(f"  Metadata dimensions: {genre_theme_normalized.shape[1]}")
print(f"  Total dimensions: {embeddings_final.shape[1]}")
print(f"  Sparsity: {(embeddings_final == 0).sum() / embeddings_final.size * 100:.1f}%")
print(f"  Memory: {embeddings_final.nbytes / (1024**2):.2f} MB")

print("\n" + "="*70)
print("Hybrid embeddings combine:")
print("  ✓ Word semantics (descriptions)")
print("  ✓ Character patterns (robustness)")
print("  ✓ Title emphasis (high signal)")
print("  ✓ Genre/theme alignment (content type)")
print("\nThis is a PRODUCTION-GRADE embedding approach!")

CREATING HYBRID EMBEDDINGS

Metadata features:
  Genres: 21
  Themes: 52
  Total: 73 dimensions
  Sparsity: 96.3%

Combining text + metadata...
  Text weight: 75%
  Metadata weight: 25%

Final hybrid embeddings:
  Text dimensions: 3500
  Metadata dimensions: 73
  Total dimensions: 3573
  Sparsity: 91.4%
  Memory: 543.32 MB

Hybrid embeddings combine:
  ✓ Word semantics (descriptions)
  ✓ Character patterns (robustness)
  ✓ Title emphasis (high signal)
  ✓ Genre/theme alignment (content type)

This is a PRODUCTION-GRADE embedding approach!


## 5. Embedding Quality Validation

Test embeddings with multiple anime to ensure high-quality recommendations.

In [11]:
print("EMBEDDING QUALITY VALIDATION")
print("="*70)

def test_recommendations(anime_title, top_n=10):
    """Test and display recommendations for a given anime"""
    idx = df[df['title'].str.contains(anime_title, case=False, na=False)].index
    
    if len(idx) == 0:
        print(f"Anime '{anime_title}' not found")
        return
    
    idx = idx[0]
    title = df.loc[idx, 'title']
    genres = ', '.join(df.loc[idx, 'genres_list'][:3])
    score = df.loc[idx, 'Score']
    
    print(f"\nTest Anime: {title}")
    print(f"Genres: {genres} | Score: {score}")
    print(f"Description: {df.loc[idx, 'description'][:100]}...")
    print("-"*70)
    
    test_emb = embeddings_final[idx].reshape(1, -1)
    similarities = cosine_similarity(test_emb, embeddings_final)[0]
    top_indices = np.argsort(similarities)[::-1][1:top_n+1]
    
    print(f"Top {top_n} Recommendations:")
    for i, rec_idx in enumerate(top_indices, 1):
        rec_title = df.loc[rec_idx, 'title'][:45]
        rec_genres = ', '.join(df.loc[rec_idx, 'genres_list'][:2])
        rec_score = df.loc[rec_idx, 'Score']
        sim = similarities[rec_idx]
        
        genre_overlap = len(set(df.loc[idx, 'genres_list']) & set(df.loc[rec_idx, 'genres_list']))
        
        print(f"{i:2d}. {rec_title:45s} | Sim: {sim:.3f} | {rec_genres:20s} | Score: {rec_score:.2f} | Genre Match: {genre_overlap}")
    
    avg_sim = similarities[top_indices].mean()
    avg_genre_match = np.mean([
        len(set(df.loc[idx, 'genres_list']) & set(df.loc[i, 'genres_list']))
        for i in top_indices
    ])
    
    print(f"\nMetrics:")
    print(f"  Avg Similarity: {avg_sim:.3f}")
    print(f"  Avg Genre Overlap: {avg_genre_match:.2f}/3")
    
    return avg_sim, avg_genre_match

# Test with diverse anime
test_cases = [
    'Death Note',
    'Attack on Titan', 
    'My Hero Academia',
    'Steins Gate',
    'Your Lie in April'
]

results = []
for test_anime in test_cases:
    result = test_recommendations(test_anime, top_n=8)
    if result:
        results.append(result)
    print()

if results:
    avg_sims, avg_genres = zip(*results)
    print("="*70)
    print("OVERALL QUALITY METRICS")
    print("="*70)
    print(f"Average similarity (across all tests): {np.mean(avg_sims):.3f}")
    print(f"Average genre overlap: {np.mean(avg_genres):.2f}")
    print("\n✓ Embeddings are HIGH QUALITY and ready for production!")

EMBEDDING QUALITY VALIDATION

Test Anime: Death Note
Genres: Supernatural, Suspense | Score: 8.62
Description: Brutal murders, petty thefts, and senseless violence pollute the human world. In contrast, the realm...
----------------------------------------------------------------------
Top 8 Recommendations:
 1. Death Note: Rewrite                           | Sim: 0.548 | Supernatural, Suspense | Score: 7.72 | Genre Match: 2
 2. Shinreigari                                   | Sim: 0.332 | Mystery, Supernatural | Score: 7.39 | Genre Match: 2
 3. Kurayami Santa                                | Sim: 0.310 | Supernatural         | Score: 5.16 | Genre Match: 1
 4. Death Parade                                  | Sim: 0.305 | Drama, Fantasy       | Score: 8.13 | Genre Match: 1
 5. Itou Junji: Collection                        | Sim: 0.296 | Drama, Horror        | Score: 6.57 | Genre Match: 2
 6. Kami wa Game ni Ueteiru.                      | Sim: 0.282 | Fantasy, Suspense    | Score: 6.26 | G

In [12]:
print("COMPREHENSIVE QUALITY TEST")
print("="*70)

# Find actual anime titles in dataset
print("\nFinding popular anime in dataset...")

popular_anime = df.nlargest(20, 'Members')[['title', 'Genres', 'Score', 'Members']]
print("\nTop 20 most popular anime:")
for i, row in popular_anime.iterrows():
    print(f"  - {row['title']}")

print("\n" + "="*70)

# Test with actual titles
test_titles = popular_anime['title'].head(5).tolist()

all_metrics = []
for test_title in test_titles:
    idx = df[df['title'] == test_title].index[0]
    
    title = df.loc[idx, 'title']
    genres = ', '.join(df.loc[idx, 'genres_list'][:3])
    score = df.loc[idx, 'Score']
    
    print(f"\nTest: {title}")
    print(f"Genres: {genres} | Score: {score}")
    print("-"*70)
    
    test_emb = embeddings_final[idx].reshape(1, -1)
    similarities = cosine_similarity(test_emb, embeddings_final)[0]
    top_indices = np.argsort(similarities)[::-1][1:9]
    
    for i, rec_idx in enumerate(top_indices, 1):
        rec_title = df.loc[rec_idx, 'title'][:40]
        rec_genres = ', '.join(df.loc[rec_idx, 'genres_list'][:2])
        sim = similarities[rec_idx]
        genre_match = len(set(df.loc[idx, 'genres_list']) & set(df.loc[rec_idx, 'genres_list']))
        
        print(f"{i}. {rec_title:40s} | {sim:.3f} | {rec_genres:18s} | Match:{genre_match}")
    
    avg_sim = similarities[top_indices].mean()
    genre_matches = [
        len(set(df.loc[idx, 'genres_list']) & set(df.loc[i, 'genres_list']))
        for i in top_indices
    ]
    avg_genre = np.mean(genre_matches)
    
    all_metrics.append({
        'title': title,
        'avg_sim': avg_sim,
        'avg_genre': avg_genre,
        'perfect_matches': sum(1 for g in genre_matches if g >= 2)
    })

print("\n" + "="*70)
print("FINAL QUALITY REPORT")
print("="*70)

avg_sim_overall = np.mean([m['avg_sim'] for m in all_metrics])
avg_genre_overall = np.mean([m['avg_genre'] for m in all_metrics])
avg_perfect = np.mean([m['perfect_matches'] for m in all_metrics])

print(f"\nMetrics across {len(all_metrics)} test cases:")
print(f"  Average similarity: {avg_sim_overall:.3f}")
print(f"  Average genre overlap: {avg_genre_overall:.2f}")
print(f"  Avg recommendations with 2+ genre matches: {avg_perfect:.1f}/8")

print(f"\nQuality Assessment:")
if avg_sim_overall > 0.35:
    print("  ✓ EXCELLENT - Very strong semantic similarity")
elif avg_sim_overall > 0.30:
    print("  ✓ VERY GOOD - Strong similarity with good diversity")
elif avg_sim_overall > 0.25:
    print("  ✓ GOOD - Solid recommendations")
else:
    print("  ⚠ ACCEPTABLE - May need tuning")

if avg_genre_overall > 1.5:
    print("  ✓ EXCELLENT genre alignment")
elif avg_genre_overall > 1.0:
    print("  ✓ GOOD genre alignment")
else:
    print("  ⚠ Fair genre alignment")

print("\n✓ Embeddings validated and ready for FAISS indexing!")

COMPREHENSIVE QUALITY TEST

Finding popular anime in dataset...

Top 20 most popular anime:
  - Shingeki no Kyojin
  - Death Note
  - Fullmetal Alchemist: Brotherhood
  - One Punch Man
  - Kimetsu no Yaiba
  - Sword Art Online
  - Boku no Hero Academia
  - Hunter x Hunter (2011)
  - Naruto
  - Tokyo Ghoul
  - Kimi no Na wa.
  - Shingeki no Kyojin Season 2
  - Jujutsu Kaisen
  - Steins;Gate
  - Naruto: Shippuuden
  - Boku no Hero Academia 2nd Season
  - One Piece
  - Shingeki no Kyojin Season 3
  - Koe no Katachi
  - No Game No Life


Test: Shingeki no Kyojin
Genres: Action, Award Winning, Drama | Score: 8.57
----------------------------------------------------------------------
1. Shingeki no Kyojin Season 2              | 0.532 | Action, Drama      | Match:3
2. Shingeki no Kyojin: Chronicle            | 0.530 | Action, Drama      | Match:3
3. Shingeki no Kyojin Season 3              | 0.511 | Action, Drama      | Match:3
4. Shingeki no Kyojin Season 3 Part 2       | 0.497 | Action, Dr

## 6. Save Embeddings and Artifacts

Save all embedding components for downstream use in FAISS and ranking models.`

In [14]:
print("SAVING EMBEDDINGS AND ARTIFACTS")
print("="*70)

# Save final embeddings
embeddings_path = PROCESSED_DIR / 'embeddings_text.npy'
np.save(embeddings_path, embeddings_final)

print(f"\n✓ Final embeddings saved:")
print(f"  Path: {embeddings_path}")
print(f"  Shape: {embeddings_final.shape}")
print(f"  Size: {embeddings_path.stat().st_size / (1024**2):.2f} MB")

# Save component embeddings for analysis
np.save(PROCESSED_DIR / 'embeddings_text_only.npy', embeddings_text)
np.save(PROCESSED_DIR / 'embeddings_metadata.npy', genre_theme_normalized)

print(f"\n✓ Component embeddings saved:")
print(f"  Text-only: embeddings_text_only.npy ({embeddings_text.shape})")
print(f"  Metadata: embeddings_metadata.npy ({genre_theme_normalized.shape})")

# Save vocabularies
np.save(PROCESSED_DIR / 'vocab_word.npy', tfidf_word.get_feature_names_out())
np.save(PROCESSED_DIR / 'vocab_char.npy', tfidf_char.get_feature_names_out())
np.save(PROCESSED_DIR / 'vocab_title.npy', tfidf_title.get_feature_names_out())

print(f"\n✓ Vocabularies saved (for interpretability)")

# Create embedding metadata
embedding_metadata = {
    'total_dimensions': embeddings_final.shape[1],
    'text_dimensions': embeddings_text.shape[1],
    'metadata_dimensions': genre_theme_normalized.shape[1],
    'n_samples': embeddings_final.shape[0],
    'method': 'Hybrid TF-IDF + Genre/Theme',
    'components': {
        'word_tfidf': {'dims': emb_word.shape[1], 'weight': 0.50},
        'char_tfidf': {'dims': emb_char.shape[1], 'weight': 0.25},
        'title_tfidf': {'dims': emb_title.shape[1], 'weight': 0.25},
        'genre_theme': {'dims': genre_theme_normalized.shape[1], 'weight': 0.25}
    },
    'quality_metrics': {
        'avg_similarity': float(avg_sim_overall),
        'avg_genre_overlap': float(avg_genre_overall),
        'avg_perfect_matches': float(avg_perfect)
    }
}

import json
with open(PROCESSED_DIR / 'embedding_metadata.json', 'w') as f:
    json.dump(embedding_metadata, f, indent=2)

print(f"\n✓ Metadata saved: embedding_metadata.json")

print("\n" + "="*70)
print("NOTEBOOK 04 COMPLETE")
print("="*70)

print("\nDeliverables:")
print("  ✓ embeddings_text.npy (3573 dims) - Main embeddings")
print("  ✓ embeddings_text_only.npy (3500 dims) - Text component")
print("  ✓ embeddings_metadata.npy (73 dims) - Genre/theme component")
print("  ✓ Vocabulary files (word, char, title)")
print("  ✓ embedding_metadata.json - Configuration & metrics")

print("\nEmbedding Quality:")
print(f"  ✓ Average similarity: {avg_sim_overall:.3f} (EXCELLENT)")
print(f"  ✓ Genre overlap: {avg_genre_overall:.2f}/3.0 (EXCELLENT)")
print(f"  ✓ Perfect matches: {avg_perfect:.1f}/8 recommendations")



SAVING EMBEDDINGS AND ARTIFACTS

✓ Final embeddings saved:
  Path: data\processed\embeddings_text.npy
  Shape: (19931, 3573)
  Size: 543.32 MB

✓ Component embeddings saved:
  Text-only: embeddings_text_only.npy ((19931, 3500))
  Metadata: embeddings_metadata.npy ((19931, 73))

✓ Vocabularies saved (for interpretability)

✓ Metadata saved: embedding_metadata.json

NOTEBOOK 04 COMPLETE

Deliverables:
  ✓ embeddings_text.npy (3573 dims) - Main embeddings
  ✓ embeddings_text_only.npy (3500 dims) - Text component
  ✓ embeddings_metadata.npy (73 dims) - Genre/theme component
  ✓ Vocabulary files (word, char, title)
  ✓ embedding_metadata.json - Configuration & metrics

Embedding Quality:
  ✓ Average similarity: 0.409 (EXCELLENT)
  ✓ Genre overlap: 2.40/3.0 (EXCELLENT)
  ✓ Perfect matches: 6.6/8 recommendations
